# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets (using their `@id`s), and preview fields and columns for each record set.

In [ ]:
# List available record sets and display their fields and columns via @id
print("Available Record Sets (by @id):")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (by @id):")
        for field in rs.fields:
            print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type if hasattr(field,'data_type') else '-'}")
            if hasattr(field, 'columns') and field.columns:
                print(f"      Columns (by @id):")
                for col in field.columns:
                    print(f"        - @id: {col.id}, name: {col.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from all record sets
dataframes = dict()
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}\n")

# Select the first record set for further demonstration
sel_record_set = record_set_ids[0] if record_set_ids else None
if sel_record_set and not dataframes[sel_record_set].empty:
    print(f"Columns in `{sel_record_set}`: {dataframes[sel_record_set].columns.tolist()}")
    display(dataframes[sel_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers or grouping key attributes for further analysis.

In [ ]:
# --- EDA ---
import numpy as np

if sel_record_set:
    df = dataframes[sel_record_set]
    print(f"Exploring record set '@id': {sel_record_set}")
    # Try to find a numeric field automatically
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field:
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Drop NA and filter for values above a threshold
        threshold = np.nanpercentile(df[numeric_field].dropna(), 75) if df[numeric_field].dtype != 'bool' else 1
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}")
        print(filtered_df.head())
        # Normalize
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
        # Try to group by another non-numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} (showing mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric field found in selected record set to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Histogram and boxplot of the selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if sel_record_set and numeric_field:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()
    # Optionally scatter with group field
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(8,4))
        sns.stripplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR² Mass Spectrometry dataset using its Croissant metadata via the `mlcroissant` library, explored its record sets (by `@id`), extracted and viewed tabular data, performed basic EDA on numeric fields, and visualized the distributions. For detailed scientific analysis or machine learning, further domain knowledge and field selection is recommended.